In [1]:
import time
import mlx.core as mx
from mlx_lm import load, generate
import mlx_lm


# 1. Modelni yuklaymiz
MODEL_ID = "mlx-community/Qwen2.5-7B-Instruct-4bit"
print("🔄 Model yuklanmoqda...")
model, tokenizer = load(MODEL_ID)
print("🚀 Model tayyor!  boshlaymiz.\n")

/Users/rakhmonberdiev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/rakhmonberdiev/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Model yuklanmoqda...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 132451.71it/s]


🚀 Model tayyor!  boshlaymiz.



In [2]:
MODEL_ID = "mlx-community/Qwen2.5-7B-Instruct-4bit"
model, tokenizer = load(MODEL_ID)

# Test uchun uzunroq kontekst matni (Sun'iy intellekt tarixi haqida ma'lumot)
U1 = "Menga sun'iy intellekt tarixi, uning o'rtaga chiqishi, 1956-yilgi Dartmut konferensiyasi va bugungi transformer arxitekturasi inqilobi haqida juda batafsil ma'lumot ber."
U2 = "Rahmat. Endi Alan Tyuring haqida qisqacha yoz."
U3 = "Yaxshi. SI qishlari nima va ular nega sodir bo'lgan?"
U4 = "Transformer modeli nechanchi yili taklif qilingan?"
U5 = "Ajoyib. Hozirgi barcha suhbatimizni 3 ta so'z bilan xulosa qil."


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 120218.90it/s]


In [3]:
system_prompt = {"role": "system", "content": "Siz qisqa javob beruvchi yordamchisiz."}

def run_clean_experiment(name, kv_cache_enabled, trim_enabled):
    print(f"\n▶️ RUNNING: {name}")
    print("-" * 50)
    
    history_messages = [system_prompt]
    turn_times = []
    questions = [U1, U2, U3, U4, U5]
    
    # Har safar tajriba boshlanishida Apple Silicon xotirasini butunlay tozalaymiz
    mx.metal.clear_cache()
    
    for idx, user_q in enumerate(questions, 1):
        if trim_enabled:
            # Rejim 2: Trimming (Faqat oxirgi 2 ta xabarni olib qolish)
            if idx == 1:
                history_messages = [system_prompt, {"role": "user", "content": user_q}]
            elif idx in [2, 3]:
                history_messages.append({"role": "user", "content": user_q})
            else:
                # Turn 4 va 5 da tarixni faqat oxirgi 4 ta element (2 user, 2 assistant gaplari) bilan cheklaymiz
                history_messages = [system_prompt] + history_messages[-4:] + [{"role": "user", "content": user_q}]
        else:
            # Rejim 1: To'liq tarix
            history_messages.append({"role": "user", "content": user_q})
            
        prompt = tokenizer.apply_chat_template(history_messages, tokenize=False, add_generation_prompt=True)
        
        # Agar KV_Cache = False bo'lsa, har bir turndan oldin GPU va MLX xotira keshini majburan o'chiramiz
        if not kv_cache_enabled:
            mx.metal.clear_cache()
            
        start_time = time.time()
        # MLX ning o'zi prompt_cache'siz ham chiziqli matnda keshni fonda ishlatadi
        response = generate(model, tokenizer, prompt=prompt, max_tokens=32, verbose=False)
        
        # Apple Silicon asinxron ishlagani uchun vaqtni aniq o'lchashga yordam beradi
        mx.eval(model.parameters()) 
        duration = time.time() - start_time
        
        turn_times.append(duration)
        print(f"Turn {idx} ketgan vaqt: {duration:.4f} soniya")
        
        history_messages.append({"role": "assistant", "content": response})
        
    return turn_times

In [4]:
# To'rtta rejimni ishga tushiramiz
times_r1_cache_on  = run_clean_experiment("Rejim 1: KV_Cache = TRUE", kv_cache_enabled=True, trim_enabled=False)
times_r1_cache_off = run_clean_experiment("Rejim 1: KV_Cache = FALSE", kv_cache_enabled=False, trim_enabled=False)
times_r2_cache_on  = run_clean_experiment("Rejim 2: KV_Cache = TRUE (Trimmed)", kv_cache_enabled=True, trim_enabled=True)
times_r2_cache_off = run_clean_experiment("Rejim 2: KV_Cache = FALSE (Trimmed)", kv_cache_enabled=False, trim_enabled=True)

from IPython.display import display, Markdown

# Oxirgi print jadvali o'rniga ushbu kodni qo'ying:
markdown_table = f"""
### 📊 YAKUNIY TAQQOSLASH JADVALI (Soniya hisobida)

| Turn | R1 (Cache ON) | R1 (Cache OFF) | R2 (Cache ON) | R2 (Cache OFF) |
| :--- | :--- | :--- | :--- | :--- |
| **Turn 1** | {times_r1_cache_on[0]:.4f} s | {times_r1_cache_off[0]:.4f} s | {times_r2_cache_on[0]:.4f} s | {times_r2_cache_off[0]:.4f} s |
| **Turn 2** | {times_r1_cache_on[1]:.4f} s | {times_r1_cache_off[1]:.4f} s | {times_r2_cache_on[1]:.4f} s | {times_r2_cache_off[1]:.4f} s |
| **Turn 3** | {times_r1_cache_on[2]:.4f} s | {times_r1_cache_off[2]:.4f} s | {times_r2_cache_on[2]:.4f} s | {times_r2_cache_off[2]:.4f} s |
| **Turn 4** | {times_r1_cache_on[3]:.4f} s | {times_r1_cache_off[3]:.4f} s | {times_r2_cache_on[3]:.4f} s | {times_r2_cache_off[3]:.4f} s |
| **Turn 5** | {times_r1_cache_on[4]:.4f} s | {times_r1_cache_off[4]:.4f} s | {times_r2_cache_on[4]:.4f} s | {times_r2_cache_off[4]:.4f} s |
"""

display(Markdown(markdown_table))


▶️ RUNNING: Rejim 1: KV_Cache = TRUE
--------------------------------------------------


mx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.


Turn 1 ketgan vaqt: 1.2248 soniya
Turn 2 ketgan vaqt: 1.0025 soniya
Turn 3 ketgan vaqt: 1.1521 soniya
Turn 4 ketgan vaqt: 1.3255 soniya
Turn 5 ketgan vaqt: 1.4678 soniya

▶️ RUNNING: Rejim 1: KV_Cache = FALSE
--------------------------------------------------
Turn 1 ketgan vaqt: 0.8871 soniya
Turn 2 ketgan vaqt: 1.0103 soniya
Turn 3 ketgan vaqt: 1.1478 soniya
Turn 4 ketgan vaqt: 1.2909 soniya
Turn 5 ketgan vaqt: 1.4447 soniya

▶️ RUNNING: Rejim 2: KV_Cache = TRUE (Trimmed)
--------------------------------------------------
Turn 1 ketgan vaqt: 0.8602 soniya
Turn 2 ketgan vaqt: 0.9978 soniya
Turn 3 ketgan vaqt: 1.1487 soniya
Turn 4 ketgan vaqt: 1.0674 soniya
Turn 5 ketgan vaqt: 0.7523 soniya

▶️ RUNNING: Rejim 2: KV_Cache = FALSE (Trimmed)
--------------------------------------------------
Turn 1 ketgan vaqt: 0.8702 soniya
Turn 2 ketgan vaqt: 1.0132 soniya
Turn 3 ketgan vaqt: 1.1528 soniya
Turn 4 ketgan vaqt: 1.0733 soniya
Turn 5 ketgan vaqt: 0.7466 soniya



### 📊 YAKUNIY TAQQOSLASH JADVALI (Soniya hisobida)

| Turn | R1 (Cache ON) | R1 (Cache OFF) | R2 (Cache ON) | R2 (Cache OFF) |
| :--- | :--- | :--- | :--- | :--- |
| **Turn 1** | 1.2248 s | 0.8871 s | 0.8602 s | 0.8702 s |
| **Turn 2** | 1.0025 s | 1.0103 s | 0.9978 s | 1.0132 s |
| **Turn 3** | 1.1521 s | 1.1478 s | 1.1487 s | 1.1528 s |
| **Turn 4** | 1.3255 s | 1.2909 s | 1.0674 s | 1.0733 s |
| **Turn 5** | 1.4678 s | 1.4447 s | 0.7523 s | 0.7466 s |


xlm_lm.generate() da kv_cache mavjud emas
